In [ ]:
# Setup: find the course materials and load our helper code. Run this cell first.
import subprocess
import sys
from pathlib import Path

# Colab has a module called google.colab; a laptop does not.
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab: download the course materials once.
    TASK_DIR = Path("/content/intern-training/tasks/esm2-enzyme-transfer")
    if not TASK_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "esm2-v1",
             "https://github.com/lagillenwater/intern-training", "/content/intern-training"],
            check=True,
        )
    # Install the exact package versions this course was tested with.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", 'accelerate==1.16.0', 'transformers==5.23.0', 'umap-learn==0.6.0'],
        check=True,
    )
else:
    # On a laptop: walk up from the current folder until we find the folder
    # that holds both helpers/ and checks.py.
    TASK_DIR = None
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "helpers").is_dir() and (folder / "checks.py").exists():
            TASK_DIR = folder
            break
    if TASK_DIR is None:
        raise RuntimeError("Could not find the course materials (a folder with helpers/ and checks.py).")

# Make our helper code importable, then load it.
sys.path.insert(0, str(TASK_DIR))
from helpers import *
from checks import *

print(f"Setup complete. Materials: {TASK_DIR}")


# 00 Python primer for Java programmers

**What this step is for:** you already know Java. This notebook teaches just enough Python to run the rest of the project, and every example uses the real enzyme data you will study. Each part puts a Python idea next to the Java you know.

**Time:** about 1.25 hours, including getting Colab set up.

**Reads:** the enzyme table, `data/enzymes.tsv`. **Writes:** nothing (it only creates your project folder on Google Drive).

**How to work through it**
- Run the cells in order, top to bottom. To run a cell, click it and press Shift+Enter.
- The Setup cell at the very top must run first, every time you open the notebook. It downloads the course materials and loads our helper code.
- Cells marked `TODO` are yours. Replace each `None` (or `# YOUR CODE HERE`) with your own code.
- The cell after each `TODO` is a check. It prints one line that starts with `Check passed` or with `Not yet` and a hint. If you see `Not yet`, fix your code and run both cells again.
- If a cell shows an error, read the last line of the message first; it usually names the problem. If the error is in a cell below a `TODO`, make sure that `TODO` passed its check.

## Configuration
This cell names the run and creates the folder where later notebooks save their results. On Colab it asks to connect your Google Drive: allow it. Results saved on Drive survive when Colab resets. This notebook saves nothing, but connecting now shows that it works.

In [ ]:
# Configuration for this run (the same in every notebook).
RUN_NAME = "esm2_8M"                # a label for this run: the small ESM-2 model, with 8 million parameters
WORK_DIR = setup_workdir(RUN_NAME)  # creates the folder (on Google Drive in Colab) and returns its path
print(WORK_DIR)

## The data
The examples use the data for this project: thousands of enzymes (proteins that speed up chemical reactions) from five species. `load_dataset()` is a function from our helper code that reads the table and gives it back. You will learn how tables work in Part 7; for now, just look.

In [ ]:
df = load_dataset()  # df is a table (a pandas DataFrame); each row is one enzyme
df.head()            # show the first five rows

The columns you will use most:
- `species`: which organism the enzyme comes from (human, mouse, fly, yeast, ecoli)
- `gene` and `protein_name`: what the enzyme is called
- `ec`: its EC number, a code for the reaction it speeds up, such as `3.4.21.4`
- `ec_class`: the first number of the EC number, 1 to 7. This is the label the model will learn to predict.
- `sequence` and `length`: the protein's chain of amino acids (one letter per amino acid) and how many there are
- `split`: whether the protein is used to train the model or held back to test it (notebook 02 explains)

## Part 1: Variables and types
In Java you declare a type: `int maxLength = 1022;`. In Python you just assign, and Python works out the type from the value. There are no semicolons, and `#` starts a comment, like `//` in Java. Python names use `snake_case` (words joined by underscores) where Java uses `camelCase`.

In [ ]:
n_proteins = len(df)      # Java: int nProteins = ...;   len() counts the rows of a table
species_name = "yeast"    # Java: String speciesName = "yeast";
fraction = 0.25           # Java: double fraction = 0.25;
is_enzyme = True          # Java: boolean isEnzyme = true;   (Python capitalizes True and False)

# print() is Java's System.out.println(). It takes several values, separated by commas.
print(n_proteins, species_name, fraction, is_enzyme)
# type() tells you what kind of value a variable holds.
print(type(n_proteins), type(species_name), type(fraction), type(is_enzyme))

Python is *dynamically typed*: the type belongs to the value, not to the variable. The same variable can hold an int now and a string later. Java would refuse to compile that. It is handy, but it also means Python finds a type mistake only when that line runs.

Also new: if the last line of a cell is a value, the notebook shows it without `print()`.

In [ ]:
x = 1022
print(type(x))
x = "now a string"
type(x)  # the last line of the cell, so its value is shown automatically

## Part 2: Lists and dicts
A Python **list** is like a Java `ArrayList`, written with square brackets. It can hold any type, and indexes start at 0, as in Java.

In [ ]:
lengths = [515, 788, 362]   # Java: ArrayList<Integer> lengths = new ArrayList<>(List.of(515, 788, 362));
print(lengths[0])           # first item (Java: lengths.get(0))
print(lengths[-1])          # last item: a negative index counts from the end
print(len(lengths))         # number of items (Java: lengths.size())
lengths.append(1022)        # add to the end (Java: lengths.add(1022))
print(lengths)

A **dict** (dictionary) is like a Java `HashMap`: it maps keys to values. It is written with curly braces, `{key: value, ...}`. Our helper code defines `SPECIES`, a dict from each species name to its taxon ID, the number biology databases use for that organism.

In [ ]:
print(SPECIES)
print(SPECIES["yeast"])    # look up one value by its key (Java: species.get("yeast"))
print("mouse" in SPECIES)  # is this key in the dict? (Java: species.containsKey("mouse"))
print(len(SPECIES))        # number of keys (Java: species.size())

**Your turn.** The five species names are the keys of `SPECIES`, printed above.

The cell after a `TODO` checks your work. It prints one line that starts with `Check passed` or `Not yet`, and under it the notebook also shows `True` or `False`: the value the check function returns, displayed because it is the last line of the cell.

In [ ]:
# TODO: Make a list named species that holds the five species names from SPECIES, each as a string.
# Hint: a list uses square brackets with items separated by commas, like ["a", "b"]; put each name in double quotes.
species = None  # replace None with your list

In [ ]:
check_species_list(species)

## Part 3: Functions
A Java method declares its return type and parameter types and wraps its body in braces. A Python function starts with `def`, ends its first line with a colon, and marks its body by **indenting** it 4 spaces. The indentation is required: it does the job of Java's braces. No types are declared.

In [ ]:
# Java:
#   static boolean isTooLong(int length) {
#       return length > MAX_RESIDUES;
#   }
def is_too_long(length):
    return length > MAX_RESIDUES  # MAX_RESIDUES (1022) is the most amino acids our model can read

print(is_too_long(515))
print(is_too_long(2000))

Strings have methods, as in Java. `split` cuts a string into a list of pieces, and `int()` turns a string of digits into a number. An EC number such as `3.4.21.4` has four parts separated by dots; the first part is the enzyme's class.

In [ ]:
ec = "2.7.11.1"
parts = ec.split(".")       # Java: ec.split("\\.")   Python splits on plain text, so the dot needs no escaping
print(parts)                # a list of four strings
print(parts[1])             # the second piece, still a string
print(int(parts[1]) + 10)   # Java: Integer.parseInt(parts[1]) + 10

In [ ]:
# TODO: Write a function first_ec_class(ec) that takes an EC number string such as "3.4.21.4"
#       and returns its class, the number before the first dot, as an int (for "3.4.21.4", that is 3).
# Hint: inside the function, use ec.split("."), take the first piece with [0], and convert it with int().
def first_ec_class(ec):
    return None  # replace None with your code

In [ ]:
# No parentheses after first_ec_class: we hand over the function itself, and the check calls it on test inputs.
check_ec_class_function(first_ec_class)

## Part 4: Imports
Python's `import` works like Java's: it loads code written elsewhere. A common habit is to give a module a short nickname with `as`. The three below are the standard tools for data work, and every notebook in this project uses them.

The Setup cell already ran `from helpers import *`. That is why `load_dataset`, `SPECIES`, and the check functions work without a module name in front.

In [ ]:
import numpy as np               # numpy: fast math on whole arrays of numbers
import pandas as pd              # pandas: tables of data, like a spreadsheet
import matplotlib.pyplot as plt  # matplotlib: plots

print(np.sqrt(16))  # reach inside a module with a dot (Java: Math.sqrt(16))

## Part 5: f-strings
Java builds text with `+` or `String.format`. A Python f-string has an `f` before the opening quote, and anything inside `{ }` is computed and inserted. A format code after a colon controls how a number looks: `:.2f` shows 2 decimal places, `:.1%` shows a percentage with 1 decimal place, and `:,` adds thousands separators.

Watch out for division: in Python, `/` always gives a decimal, so `2 / 3` is `0.666...`. Java's `2 / 3` with ints gives `0`.

In [ ]:
n_species = len(SPECIES)  # len() of a dict counts its keys
print(f"The table has {len(df):,} enzymes from {n_species} species.")  # Java: "The table has " + n + " enzymes"
print(f"The model reads at most {MAX_RESIDUES} amino acids.")
print(f"Two decimals: {2 / 3:.2f}   As a percent: {2 / 3:.1%}")

## Part 6: numpy arrays and vectorized math
A numpy array is like a Java array of numbers, with one big difference: math and comparisons apply to **every item at once**, without a loop. This is called *vectorized* code. It is shorter, and much faster, because the looping happens inside numpy's fast compiled code.

In [ ]:
lengths = df["length"].to_numpy()  # the length column of the table, as a numpy array (the list from Part 2 is replaced)
print(type(lengths))
print(lengths.shape)               # the array's size, like Java's lengths.length
print(lengths[:5])                 # a "slice": items from index 0 up to, but not including, index 5

In Java you would add up all the lengths with a loop. Python can loop too (shown once here, so you can compare), but with numpy you rarely need to.

In [ ]:
# The Java way, written in Python. "for x in items:" is Java's for-each loop, for (int x : items).
total = 0
for length in lengths:
    total += length
print(total)

# The numpy way: one call, no loop.
print(lengths.sum())

In [ ]:
# Math on an array happens to every item at once.
# An amino acid weighs about 110 daltons, so this estimates every protein's mass in kilodaltons.
mass_kda = lengths * 110 / 1000
print(mass_kda[:5])
print(lengths.mean(), lengths.max())  # summaries of the whole array: the average and the largest value

**Boolean masks.** A comparison on an array is also done item by item. The result is an array of `True`/`False` values, called a *mask*. Putting a mask inside `[ ]` keeps only the items where the mask is `True`. And because `True` counts as 1 and `False` as 0, `.sum()` on a mask counts the `True`s.

In [ ]:
is_long = lengths > 1000     # True where the protein has more than 1000 amino acids
print(is_long[:5])
print(is_long.sum())         # how many proteins have more than 1000
print(lengths[is_long][:5])  # keep only the long ones, then show the first five

In [ ]:
# A mask made from one column can select items from another, because both arrays are in the same row order.
species_col = df["species"].to_numpy()  # the species of each protein, lined up with lengths
is_human = species_col == "human"       # True for human proteins
print(lengths[is_human].max())          # the longest human protein

In [ ]:
# TODO: Compute the mean (average) length of the yeast proteins. Store it in yeast_mean_length.
# Hint: make a mask with species_col == "yeast", use it to select from lengths, then call .mean().
yeast_mean_length = None  # replace None with your code

In [ ]:
check_mean_length(yeast_mean_length, df)

## Part 7: pandas tables
A pandas **DataFrame** is a table: named columns and one row per item. `df` is one; think of it as a spreadsheet you control with code. Inside `load_dataset()` is a call to `pd.read_csv`, which reads a text file of rows and columns. Our file is tab-separated (a TSV file), so we pass `sep="\t"`.

In [ ]:
path = task_dir() / "data" / "enzymes.tsv"  # the / joins folder names into a file path
table = pd.read_csv(path, sep="\t")         # read the tab-separated file into a DataFrame
print(table.shape)                          # (number of rows, number of columns)
print(list(table.columns))                  # the column names, as a list

From here on we use `df`, which `load_dataset()` also tidied up a little. One column of a table is a **Series**: its values, plus a row label (the *index*) on the left. Select one column with `df["name"]`, or several with a list of names, `df[["name1", "name2"]]`. You may also see `df.name`, a shortcut for `df["name"]`.

In [ ]:
df[["gene", "species", "ec", "length"]].head()  # a smaller table with just these four columns

**Filtering rows** works like a numpy mask: compare a column to get `True`/`False` for each row, then put that inside `df[ ]`. To combine two conditions, use `&` (and) or `|` (or), with each condition in its own parentheses. Java uses `&&` and `||`, but pandas needs `&` and `|`.

In [ ]:
ecoli = df[df["species"] == "ecoli"]                                 # only the E. coli rows
long_ecoli = df[(df["species"] == "ecoli") & (df["length"] > 1000)]  # E. coli rows that are also long
print(len(ecoli), len(long_ecoli))
long_ecoli[["gene", "protein_name", "length"]]

**Counting.** `value_counts()` counts how often each value appears in a column; in Java this would be a loop filling a `HashMap`. Here it counts the `split` column, which marks the proteins the model trains on (`train`), the ones used to tune it (`val`), and the ones held back to test it (`test`).

In [ ]:
df["split"].value_counts()  # how many rows have each value of split

**Grouping.** `groupby` splits the rows into groups (here, one per species), computes something within each group, and collects the answers in one Series. Find yeast in the output: it should match your `yeast_mean_length` from Part 6.

In [ ]:
mean_length = df.groupby("species")["length"].mean()  # the mean length within each species
mean_length

In [ ]:
# TODO: Count how many proteins each species has. Store the result in species_counts.
# Hint: use .value_counts() on the species column, df["species"].
species_counts = None  # replace None with your code

In [ ]:
check_species_counts(species_counts, df)

## Part 8: Plots with matplotlib
`plt.bar(labels, heights)` draws one bar per label. A Series holds both: `.index` has the labels and `.values` has the numbers. Label your axes and add a title, so a reader knows what they are looking at. `plt.show()` draws the finished plot.

In [ ]:
plt.bar(mean_length.index, mean_length.values, color="#2a78d6")  # the color is a hex code, as on web pages
plt.xlabel("Species")
plt.ylabel("Mean length (amino acids)")
plt.title("Mean enzyme length by species")
plt.show()

In [ ]:
# TODO: Draw a bar chart of species_counts, with axis labels and a title.
# Hint: follow the cell above: plt.bar(species_counts.index, species_counts.values), then plt.xlabel, plt.ylabel, plt.title, and plt.show().
# YOUR CODE HERE

## You are ready
That is all the Python this project needs. Keep this table handy:

| Idea | Java | Python |
|---|---|---|
| variable | `int n = 5;` | `n = 5` |
| comment | `// note` | `# note` |
| list | `ArrayList<Integer> xs` | `xs = [1, 2, 3]` |
| map | `HashMap<String, Integer> m` | `m = {"a": 1}` |
| function | `static int f(int x) { ... }` | `def f(x):` and an indented body |
| text with values | `"n = " + n` | `f"n = {n}"` |
| loop over items | `for (int x : xs)` | `for x in xs:` (but prefer vectorized numpy and pandas) |
| keep matching rows | a loop with an `if` | `df[df["species"] == "yeast"]` |

For "and" and "or": Java's `&&` and `||` become the words `and` and `or` for single True/False values, and `&` and `|` for masks and table columns.

Next: notebook 01, where you explore the data.